# Phase 245: Re-embed the COACH420/HOLO4K benchmark cache at ESM2-3B

Decodes the amino-acid sequence losslessly from Phase 15's cached one-hot block (no re-fetch, no re-parse),
re-embeds every chain at ESM2-3B (3B params, hidden 2560, fp16, batch 1), and reassembles
`[onehot(21) | dihedral(6) | ESM-3B | topo-raw(34)]` -- a drop-in replacement for
`phase15_raw_features.pkl` at this embedding scale, for the localization geometry control (Phase 246/247).

## Cell 1: Install

In [ ]:
!pip install -q biopython==1.84 transformers==4.46.3 accelerate

## Cell 2: Load Phase 15's cache, decode sequences from the one-hot block

In [ ]:
import pickle, time
from pathlib import Path
import numpy as np
import torch

def locate(f):
    c=[]
    for r in (Path("/kaggle/input"), Path("/kaggle/working"), Path(".")):
        if r.exists(): c.extend(r.rglob(f))
    if not c: raise FileNotFoundError(f)
    return max(c, key=lambda p: p.stat().st_size)

with open(locate("phase15_raw_features.pkl"), "rb") as fh:
    raw_cache = pickle.load(fh)

AA_ALPHABET = "ACDEFGHIKLMNPQRSTVWYX"          # identical ordering to Phase 15's own A2I
N_PRE_OLD, TOPO_DIM = 507, 34                   # 21 onehot + 6 dihedral + 480 ESM-35M, then 34 raw topo
assert N_PRE_OLD == 21 + 6 + 480

seqs = {}
for k, e in raw_cache.items():
    oh = e["x"][:, :21]
    idx = oh.argmax(axis=1)
    assert np.allclose(oh[np.arange(len(idx)), idx], 1.0), f"{k}: onehot block is not a clean one-hot"
    seqs[k] = "".join(AA_ALPHABET[i] for i in idx)

print(f"{len(raw_cache):,} cached chains; decoded {len(seqs):,} sequences "
      f"({sum(len(s) for s in seqs.values()):,} residues total)")

GPU_OK=False
if torch.cuda.is_available():
    try:
        _p=torch.randn(256,256,device="cuda"); (_p@_p).sum().item(); torch.cuda.synchronize(); GPU_OK=True
        print(f"GPU: {torch.cuda.get_device_name(0)} (probe PASS)")
    except RuntimeError as e: print("GPU unusable:", e)
DEVICE = torch.device("cuda" if GPU_OK else "cpu")

## Cell 3: ESM2-3B embeddings, fused with reassembly (memory-lean: never holds a full-size copy of the embeddings twice)

In [ ]:
import gc
from transformers import AutoTokenizer, AutoModel

ESM_NAME = "facebook/esm2_t36_3B_UR50D"   # 3B params, hidden 2560, fp16, batch 1
tok = AutoTokenizer.from_pretrained(ESM_NAME)
esm = AutoModel.from_pretrained(ESM_NAME).half().to(DEVICE).eval()
ESM_DIM = esm.config.hidden_size
print(f"{ESM_NAME}: hidden {ESM_DIM}, device {DEVICE}, dtype {next(esm.parameters()).dtype}")

@torch.no_grad()
def embed_batch(chunk_seqs, max_len=1002):
    enc = tok(chunk_seqs, return_tensors="pt", padding=True, truncation=True, max_length=max_len)
    enc = {k: v.to(DEVICE) for k, v in enc.items()}
    out = esm(**enc).last_hidden_state
    embs = []
    for i, s in enumerate(chunk_seqs):
        n = len(s)
        embs.append(out[i, 1:1+n, :].float().cpu().numpy())
    return embs

# Reassembly is fused into the same loop as embedding: each chain's embedding is consumed and
# discarded immediately, so peak memory never holds a second full-size copy of the embedding matrix
# (Phase 244/245's first attempt built a separate `new_esm` dict for ALL chains, then a `new_cache`
# duplicating it again inside the concatenated x arrays -- that held 2 full copies simultaneously
# and OOM-killed the kernel silently after finishing all real work, right before the save).
keys = sorted(seqs, key=lambda k: len(seqs[k]))
BATCH = 1
t0 = time.time()
new_cache = {}
n_bad = 0
for bi in range(0, len(keys), BATCH):
    chunk = keys[bi:bi+BATCH]
    embs = embed_batch([seqs[k] for k in chunk])
    for k, esm_e in zip(chunk, embs):
        e = raw_cache[k]
        # BUG (fixed): this used to slice dihedral as [:, 21:N_PRE_OLD], which is 21:507 -- that
        # captures dihedral(6) PLUS the old ESM-35M embedding(480) together, not just dihedral.
        # dihedral is exactly 6 columns, always at [21:27]; [27:N_PRE_OLD] is the old ESM block, discarded.
        onehot = e["x"][:, :21]; dihedral = e["x"][:, 21:27]; topo = e["x"][:, N_PRE_OLD:]
        if esm_e.shape[0] != onehot.shape[0] or not np.isfinite(esm_e).all():
            n_bad += 1; continue
        # stored as float16: halves memory (the model casts back up on load), and the source values
        # are already fp16-precision (the embedding came straight off a fp16 model)
        x_new = np.concatenate([onehot, dihedral, esm_e, topo], axis=1).astype(np.float16)
        entry = dict(e); entry["x"] = x_new
        new_cache[k] = entry
        e["x"] = None   # old (smaller) ESM-35M array no longer needed -- shrink raw_cache as we go
    if bi % (BATCH*200) == 0:
        print(f"  {bi}/{len(keys)}  [{(time.time()-t0)/60:.1f}m]")
    if bi % (BATCH*400) == 0:
        gc.collect()

n_res = sum(len(s) for s in seqs.values())
print(f"\nembedded+reassembled {len(new_cache):,}/{len(keys):,} chains / {n_res:,} residues "
      f"in {(time.time()-t0)/60:.1f} min ({n_bad} failed alignment/finite check)")
assert n_bad == 0, "some chains failed to reassemble -- investigate before Phase 246/247"
del esm, raw_cache, seqs; torch.cuda.empty_cache(); gc.collect()

import resource
peak_mb = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1024
print(f"peak RSS so far: {peak_mb:,.0f} MB")

## Cell 4: Save (float16, in two shards by benchmark source, to keep any single write's peak memory down)

In [ ]:
import resource
for src in ("coach420", "holo4k"):
    shard = {k: v for k, v in new_cache.items() if k[0] == src}
    torch.save(dict(cache=shard, esm_dim=int(ESM_DIM), n_pre=21+6+int(ESM_DIM), topo_dim=TOPO_DIM),
               f"/kaggle/working/phase245_raw_features_esm3b.{src}.pt")
    print(f"saved {len(shard):,} chains -> /kaggle/working/phase245_raw_features_esm3b.{src}.pt  "
          f"(peak RSS {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1024:,.0f} MB)")